In [1]:
%pip install -q langchain langchain-core langchain-community langchain-experimental pydantic duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


## Built-in Tool - DuckDuckGo Search

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in usa today')

print(results)

An investigation is underway and officials will release the name of the officer who fired his weapon, the police chief said. NBC News’ Shaquille Brewster on what we know about the Madison victim 6 hours ago - TOP STORIES · Sex work or starve: Aid workers in Nepal left jobless by USAID cuts turn to sex work to survive · US military launches new strikes on Iran as clashes escalate over shipping routes · How protests over a popular minister's firing sparked Ukraine's sudden military reset · Newsletters · 16 hours ago - The World Cup has captured the imagination of fans across the United States, no better illustrated than in the tournament store in Houston, where locals have been snapping up mementos of a rare chance to have the global finals on their doorstep. ... A U.S. appeals court on Thursday lifted a judge's order requiring President Donald Trump's administration to reinstall dozens of exhibits that it removed from national parks on topics such as slavery and climate change. 10 hours

In [5]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-in Tool - Shell Tool

In [9]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('whoami')

print(results)

Executing command:
 whoami
aiml



## Custom Tools

In [10]:
from langchain_core.tools import tool

In [19]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [12]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [21]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [22]:
result = multiply.invoke({"a":3, "b":5})

In [23]:
print(result)

15


In [24]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [25]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using StructuredTool

In [35]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [36]:
class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [37]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [38]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [39]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [40]:
from langchain_core.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

In [41]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [32]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [33]:
multiply_tool = MultiplyTool()

In [34]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [42]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [43]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [44]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers


In [45]:
# tools[0] is add
print(tools[0].invoke({"a": 10, "b": 20}))

# tools[1] is multiply
print(tools[1].invoke({"a": 10, "b": 20}))

30
200
